# Human-in-the-Loop: Approve before posting

The LLM drafts a LinkedIn post, then the graph **pauses** and waits for a human to approve,
reject or edit it. Only then does it continue.

```
START -> write_draft -> human_review (⏸ interrupt) --approve/edit--> publish -> END
                                                   --reject-------> END
```

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
class PostState(TypedDict):
    topic: str
    draft: str
    decision: str
    published: str

In [3]:
def write_draft(state: PostState):
    prompt = f"Write a short LinkedIn post (max 3 lines, no hashtags) about: {state['topic']}"
    return {'draft': model.invoke(prompt).content}


def human_review(state: PostState):
    # interrupt() PAUSES the graph here. Whatever we pass in is shown to the human.
    # When the graph is resumed with Command(resume=...), that value is returned here.
    answer = interrupt({'question': 'Approve this post? (approve / reject / or type an edited post)',
                        'draft': state['draft']})

    if answer == 'approve':
        return {'decision': 'approved'}
    if answer == 'reject':
        return {'decision': 'rejected'}
    return {'decision': 'edited', 'draft': answer}  # anything else = the human's edited text


def publish(state: PostState):
    return {'published': state['draft']}  # in a real app: call the LinkedIn API here


def route_decision(state: PostState):
    return END if state['decision'] == 'rejected' else 'publish'

In [4]:
graph = StateGraph(PostState)

graph.add_node('write_draft', write_draft)
graph.add_node('human_review', human_review)
graph.add_node('publish', publish)

graph.add_edge(START, 'write_draft')
graph.add_edge('write_draft', 'human_review')
graph.add_conditional_edges('human_review', route_decision, ['publish', END])
graph.add_edge('publish', END)

# A checkpointer is REQUIRED: the paused state must be saved somewhere while we wait for the human
workflow = graph.compile(checkpointer=InMemorySaver())

In [5]:
# Run 1: the graph runs until interrupt() and stops
config = {'configurable': {'thread_id': 'post-1'}}

result = workflow.invoke({'topic': 'finishing my first LangGraph project'}, config=config)

print('Paused with:', result['__interrupt__'][0].value)
print('Next node to run:', workflow.get_state(config).next)

Paused with: {'question': 'Approve this post? (approve / reject / or type an edited post)', 'draft': 'Just wrapped up my first LangGraph project—turning raw data into a conversational AI that actually understands context.  \nThe learning curve was steep, but the payoff is a seamless, reusable workflow that scales with any domain.  \nExcited to explore how this foundation can power smarter, more human‑centric applications.'}
Next node to run: ('human_review',)


In [6]:
# Resume: Command(resume=...) sends the human's answer back into interrupt()
result = workflow.invoke(Command(resume='approve'), config=config)

print('Decision:', result['decision'])
print('Published:', result['published'])

Decision: approved
Published: Just wrapped up my first LangGraph project—turning raw data into a conversational AI that actually understands context.  
The learning curve was steep, but the payoff is a seamless, reusable workflow that scales with any domain.  
Excited to explore how this foundation can power smarter, more human‑centric applications.


In [7]:
# Another run where the human edits the draft instead
config = {'configurable': {'thread_id': 'post-2'}}

workflow.invoke({'topic': 'learning LangGraph'}, config=config)
result = workflow.invoke(Command(resume='Learning LangGraph one topic at a time. Today: human-in-the-loop.'), config=config)

print('Decision:', result['decision'])
print('Published:', result['published'])

Decision: edited
Published: Learning LangGraph one topic at a time. Today: human-in-the-loop.


In [8]:
# And a rejected one: the graph ends without publishing
config = {'configurable': {'thread_id': 'post-3'}}

workflow.invoke({'topic': 'why I love Mondays'}, config=config)
result = workflow.invoke(Command(resume='reject'), config=config)

print('Decision:', result['decision'])
print('Published:', result.get('published', '(nothing published)'))

Decision: rejected
Published: (nothing published)


In [ ]:
# Try it yourself: you are the human reviewer
config = {'configurable': {'thread_id': 'post-you'}}

result = workflow.invoke({'topic': input('Topic: ')}, config=config)
print('\nDraft:\n', result['__interrupt__'][0].value['draft'])

result = workflow.invoke(Command(resume=input('\napprove / reject / or type your edit: ')), config=config)
print('\nDecision:', result['decision'], '| Published:', result.get('published', '(nothing)'))